# Reconstruction Error by Variable

This notebook analyses reconstruction error at the variable level, comparing Autoencoder (AE) and PCA reconstructions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textwrap import fill

# Set seaborn style
sns.set(style="white")

# Color palette (consistent with other notebooks)
palette = {
    "AE": "seagreen",
    "PCA": "sandybrown"
}

## Configuration

In [ ]:
# Paths
DATA_PATH = "../data/census_data/engcensus_cleaned_scaled.parquet"
AE_OUTPUT_PATH = "../AE_outputs/engcensus_all/250epoch_scan_lin"
PCA_OUTPUT_PATH = "../AE_outputs/engcensus_all/PCA"
CATEGORY_PATH = "../data/census_data/Category.csv"
METADATA_PATH = "../data/census_data/Variable_Metadata.csv"
PLOT_OUTPUT_PATH = "plots"

# Parameters
BOTTLENECK = 100

## Load Data

In [ ]:
# Load original data
data = pd.read_parquet(DATA_PATH)
data = data.set_index(data.columns[0])  # Set OA as index

# Load reconstructed autoencoder data
reco_path = f"{AE_OUTPUT_PATH}/census_geodemo__bottleneck_{BOTTLENECK}_v1__reconstructed_outputs.csv"
reco_data = pd.read_csv(reco_path, index_col=0)

# Load PCA reconstruction data
pca_path = f"{PCA_OUTPUT_PATH}/{BOTTLENECK}_components.csv"
pca_reco = pd.read_csv(pca_path, index_col=0)

# Load category mapping
cat = pd.read_csv(CATEGORY_PATH)

# Load variable metadata
metadata = pd.read_csv(METADATA_PATH)

# Create lookup dictionaries from metadata
# Table names lookup: ts001 -> "Residence type"
TABLE_NAMES = metadata[["Table_ID", "Table_Name"]].drop_duplicates().set_index("Table_ID")["Table_Name"].to_dict()

# Variable names lookup: ts0010002 -> "Residence type: Lives in a household"
VAR_NAMES = metadata.set_index("Variable_ID")["Variable_Name"].to_dict()

print(f"Original data shape: {data.shape}")
print(f"AE reconstructed shape: {reco_data.shape}")
print(f"PCA reconstructed shape: {pca_reco.shape}")
print(f"Loaded {len(TABLE_NAMES)} table names and {len(VAR_NAMES)} variable names from metadata")

## Compute Variable-Level Reconstruction Errors

In [ ]:
# Compute mean absolute reconstruction errors per variable
ae_err_var = np.abs(data - reco_data).mean(axis=0)
pca_err_var = np.abs(data - pca_reco).mean(axis=0)

# Clean variable names (remove _PCT suffix)
ae_err_var.index = ae_err_var.index.str.replace("_PCT", "")
pca_err_var.index = pca_err_var.index.str.replace("_PCT", "")

# Compute percentage difference: (AE - PCA) / PCA * 100
perc_diff = (ae_err_var - pca_err_var) / pca_err_var * 100

print(f"Mean AE reconstruction error: {ae_err_var.mean():.6f}")
print(f"Mean PCA reconstruction error: {pca_err_var.mean():.6f}")
print(f"Mean % difference (AE-PCA)/PCA: {perc_diff.mean():.2f}%")
print(f"Variables where AE is better: {(perc_diff < 0).sum()} / {len(perc_diff)} ({(perc_diff < 0).sum()/len(perc_diff)*100:.1f}%)")

## Variable-Level Error Comparison

In [ ]:
# Plot percentage difference by variable
plt.figure(figsize=(14, 5))
bars = plt.bar(range(len(perc_diff)), perc_diff.values, width=0.8)

# Color bars based on which method is better
for bar, diff in zip(bars, perc_diff.values):
    if diff < 0:
        bar.set_color(palette["AE"])  # AE better
    else:
        bar.set_color(palette["PCA"])  # PCA better

plt.xlabel("Variable", fontsize=12)
plt.ylabel("(AE - PCA) / PCA [%]", fontsize=12)
plt.title(f"Reconstruction Error % Difference by Variable (Bottleneck={BOTTLENECK})\nGreen = AE better, Orange = PCA better", fontsize=14)
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.axhline(perc_diff.mean(), color='red', linestyle='--', linewidth=1, label=f'Mean: {perc_diff.mean():.1f}%')
plt.xticks([])  # Too many variables to show labels
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(f"{PLOT_OUTPUT_PATH}/error_perc_diff_by_variable_bottleneck_{BOTTLENECK}.png", dpi=300)
plt.show()

In [ ]:
# Helper function to get variable name from code
def get_var_name(var_code):
    """Get variable name from metadata lookup."""
    return VAR_NAMES.get(var_code, "Unknown")

def get_table_name(var_code):
    """Extract table code and return table name."""
    # Handle both ts001 and ts007a formats
    if len(var_code) > 5 and var_code[5:6].isalpha():
        table = var_code[:6]  # e.g., ts007a
    else:
        table = var_code[:5]  # e.g., ts001
    return TABLE_NAMES.get(table, "Unknown")

# Print top 10 variables where AE has higher error than PCA (largest positive % diff)
top_10 = perc_diff.nlargest(10)
print("Top 10 Variables where AE Error > PCA Error (worst for AE):")
for var, diff in top_10.items():
    print(f"  {var}: {diff:+.1f}%")
    print(f"    -> {get_var_name(var)}")
print()

# Print bottom 10 variables where AE has lower error than PCA (largest negative % diff)
bottom_10 = perc_diff.nsmallest(10)
print("Top 10 Variables where AE Error < PCA Error (best for AE):")
for var, diff in bottom_10.items():
    print(f"  {var}: {diff:+.1f}%")
    print(f"    -> {get_var_name(var)}")

## Table-Level Error Comparison

In [ ]:
# Extract table codes from variable names (e.g., ts001002 -> ts001)
ae_err_table = ae_err_var.copy()
pca_err_table = pca_err_var.copy()

ae_err_table.index = ae_err_table.index.str[:-4]  # Remove last 4 chars (variable number)
pca_err_table.index = pca_err_table.index.str[:-4]

# Group by table and compute mean errors
ae_err_by_table = ae_err_table.groupby(ae_err_table.index).mean()
pca_err_by_table = pca_err_table.groupby(pca_err_table.index).mean()

# Create comparison DataFrame
table_comparison = pd.DataFrame({
    "AE": ae_err_by_table,
    "PCA": pca_err_by_table
})

# Compute percentage difference: (AE - PCA) / PCA * 100
table_comparison["Perc_Diff"] = (table_comparison["AE"] - table_comparison["PCA"]) / table_comparison["PCA"] * 100

# Add table names
table_comparison["Table Name"] = table_comparison.index.map(TABLE_NAMES)
table_comparison["Label"] = table_comparison.index + ": " + table_comparison["Table Name"].fillna("Unknown")

print(f"Number of tables: {len(table_comparison)}")
print(f"\nMean % difference across tables: {table_comparison['Perc_Diff'].mean():.1f}%")
print(f"Tables where AE is better: {(table_comparison['Perc_Diff'] < 0).sum()} / {len(table_comparison)}")
print("\nTable codes, names, and % difference:")
print(table_comparison[["Table Name", "Perc_Diff"]].round(1).to_string())

In [ ]:
# Create side-by-side plot like notebook 4
fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# Top panel: AE vs PCA mean errors
x = np.arange(len(table_comparison))
width = 0.35

axes[0].bar(x - width/2, table_comparison['AE'] * 100, width, label='AE', color=palette['AE'], edgecolor='black')
axes[0].bar(x + width/2, table_comparison['PCA'] * 100, width, label='PCA', color=palette['PCA'], edgecolor='black')
axes[0].set_ylabel('Mean Reconstruction Error (%)', fontsize=12)
axes[0].set_title(f'Reconstruction Error by Table (Bottleneck={BOTTLENECK})', fontsize=14, fontweight='bold')
axes[0].legend(frameon=False, fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Bottom panel: Percentage difference
bars = axes[1].bar(x, table_comparison['Perc_Diff'], color='darkblue', edgecolor='black')

# Color bars based on which method is better
for bar, diff in zip(bars, table_comparison['Perc_Diff']):
    if diff < 0:
        bar.set_color(palette["AE"])
    else:
        bar.set_color(palette["PCA"])

axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].axhline(table_comparison['Perc_Diff'].mean(), color='red', linestyle='--', 
                label=f'Mean: {table_comparison["Perc_Diff"].mean():.1f}%')
axes[1].set_xlabel('Table', fontsize=12)
axes[1].set_ylabel('(AE-PCA)/PCA [%]', fontsize=12)
axes[1].legend(frameon=False)
axes[1].grid(True, alpha=0.3, axis='y')

# Set x-tick labels
axes[1].set_xticks(x)
axes[1].set_xticklabels(table_comparison['Label'], rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig(f"{PLOT_OUTPUT_PATH}/error_by_table_bottleneck_{BOTTLENECK}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Print tables sorted by percentage difference
print("Tables where AE has lower error (sorted by improvement):")
ae_better = table_comparison[table_comparison["Perc_Diff"] < 0].sort_values("Perc_Diff")
for idx, row in ae_better.iterrows():
    print(f"  {idx}: {row['Table Name']} ({row['Perc_Diff']:+.1f}%)")

print(f"\nTables where PCA has lower error (sorted by improvement):")
pca_better = table_comparison[table_comparison["Perc_Diff"] > 0].sort_values("Perc_Diff", ascending=False)
for idx, row in pca_better.iterrows():
    print(f"  {idx}: {row['Table Name']} ({row['Perc_Diff']:+.1f}%)")

## Category-Level Error Comparison

In [ ]:
# Merge table errors with category information
ae_cat = table_comparison[["AE", "Table Name"]].reset_index()
ae_cat.columns = ["Table", "AE", "Table Name"]
ae_cat = ae_cat.merge(cat, on="Table", how="left")

pca_cat = table_comparison[["PCA"]].reset_index()
pca_cat.columns = ["Table", "PCA"]
pca_cat = pca_cat.merge(cat, on="Table", how="left")

# Group by category
category_ae = ae_cat.groupby("Category")["AE"].mean()
category_pca = pca_cat.groupby("Category")["PCA"].mean()

# Create category comparison DataFrame
category_comparison = pd.DataFrame({
    "AE": category_ae,
    "PCA": category_pca
})

# Compute percentage difference: (AE - PCA) / PCA * 100
category_comparison["Perc_Diff"] = (category_comparison["AE"] - category_comparison["PCA"]) / category_comparison["PCA"] * 100

print("Category-level comparison:")
print(category_comparison[["AE", "PCA", "Perc_Diff"]].round(4).to_string())
print(f"\nMean % difference across categories: {category_comparison['Perc_Diff'].mean():.1f}%")

In [ ]:
# Create side-by-side plot like notebook 4
fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# Top panel: AE vs PCA mean errors
x = np.arange(len(category_comparison))
width = 0.35

axes[0].bar(x - width/2, category_comparison['AE'] * 100, width, label='AE', color=palette['AE'], edgecolor='black')
axes[0].bar(x + width/2, category_comparison['PCA'] * 100, width, label='PCA', color=palette['PCA'], edgecolor='black')
axes[0].set_ylabel('Mean Reconstruction Error (%)', fontsize=12)
axes[0].set_title(f'Reconstruction Error by Category (Bottleneck={BOTTLENECK})', fontsize=14, fontweight='bold')
axes[0].legend(frameon=False, fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Bottom panel: Percentage difference
bars = axes[1].bar(x, category_comparison['Perc_Diff'], color='darkblue', edgecolor='black')

# Color bars based on which method is better
for bar, diff in zip(bars, category_comparison['Perc_Diff']):
    if diff < 0:
        bar.set_color(palette["AE"])
    else:
        bar.set_color(palette["PCA"])

axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].axhline(category_comparison['Perc_Diff'].mean(), color='red', linestyle='--',
                label=f'Mean: {category_comparison["Perc_Diff"].mean():.1f}%')
axes[1].set_xlabel('Category', fontsize=12)
axes[1].set_ylabel('(AE-PCA)/PCA [%]', fontsize=12)
axes[1].legend(frameon=False)
axes[1].grid(True, alpha=0.3, axis='y')

# Set x-tick labels
axes[1].set_xticks(x)
wrapped_labels = [fill(label, width=15) for label in category_comparison.index]
axes[1].set_xticklabels(wrapped_labels, rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig(f"{PLOT_OUTPUT_PATH}/error_by_category_bottleneck_{BOTTLENECK}.png", dpi=300, bbox_inches='tight')
plt.show()

## Error vs Mean Value Relationship

In [ ]:
# Examine relationship between variable mean and reconstruction error
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# AE error vs mean
axes[0].scatter(data.mean(axis=0), ae_err_var, alpha=0.5, color=palette["AE"])
axes[0].set_xlabel("Variable Mean", fontsize=12)
axes[0].set_ylabel("AE Reconstruction Error", fontsize=12)
axes[0].set_title("AE Error vs Variable Mean", fontsize=14)

# PCA error vs mean
axes[1].scatter(data.mean(axis=0), pca_err_var, alpha=0.5, color=palette["PCA"])
axes[1].set_xlabel("Variable Mean", fontsize=12)
axes[1].set_ylabel("PCA Reconstruction Error", fontsize=12)
axes[1].set_title("PCA Error vs Variable Mean", fontsize=14)

plt.tight_layout()
plt.savefig(f"{PLOT_OUTPUT_PATH}/error_vs_mean_bottleneck_{BOTTLENECK}.png", dpi=300)
plt.show()

## Summary Statistics

In [ ]:
# Summary statistics
print("=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

# Variable level
n_vars_ae_better = (perc_diff < 0).sum()
n_vars_pca_better = (perc_diff > 0).sum()
n_vars = len(perc_diff)

print(f"\nVariable Level ({n_vars} variables):")
print(f"  Variables where AE is better: {n_vars_ae_better} ({n_vars_ae_better/n_vars*100:.1f}%)")
print(f"  Variables where PCA is better: {n_vars_pca_better} ({n_vars_pca_better/n_vars*100:.1f}%)")
print(f"  Mean % difference: {perc_diff.mean():.1f}%")

# Table level
n_tables_ae_better = (table_comparison["Perc_Diff"] < 0).sum()
n_tables_pca_better = (table_comparison["Perc_Diff"] > 0).sum()
n_tables = len(table_comparison)

print(f"\nTable Level ({n_tables} tables):")
print(f"  Tables where AE is better: {n_tables_ae_better} ({n_tables_ae_better/n_tables*100:.1f}%)")
print(f"  Tables where PCA is better: {n_tables_pca_better} ({n_tables_pca_better/n_tables*100:.1f}%)")
print(f"  Mean % difference: {table_comparison['Perc_Diff'].mean():.1f}%")

# Category level
n_cats_ae_better = (category_comparison["Perc_Diff"] < 0).sum()
n_cats_pca_better = (category_comparison["Perc_Diff"] > 0).sum()
n_cats = len(category_comparison)

print(f"\nCategory Level ({n_cats} categories):")
print(f"  Categories where AE is better: {n_cats_ae_better} ({n_cats_ae_better/n_cats*100:.1f}%)")
print(f"  Categories where PCA is better: {n_cats_pca_better} ({n_cats_pca_better/n_cats*100:.1f}%)")
print(f"  Mean % difference: {category_comparison['Perc_Diff'].mean():.1f}%")

print("=" * 80)